# Commodity Data Collector (Parameterized)

This notebook combines web scraping of commodity-related news, downloading historical price data from FRED, and preprocessing to extract dense embeddings and sentiment scores using FinBERT.

**Configuration:** Edit the cell below to specify which commodity, news source, and whether to include FRED data.

## Configuration Cell

**EDIT THIS CELL TO RUN YOUR DESIRED CONFIGURATION**

In [ ]:
# ============================================================================
# USER CONFIGURATION
# ============================================================================

# Which commodity to process: 'wheat', 'corn', or 'oil'
SELECTED_COMMODITY = 'wheat'

# Which news collection method: 'direct', 'payload', or 'bloomberg'
# - 'direct': Scrape directly from Investing.com pages
# - 'payload': Scrape Investing.com payload JSON files
# - 'bloomberg': Fetch Bloomberg news via SERP API (requires API keys)
SELECTED_NEWS_TYPE = 'direct'

# Whether to include FRED macroeconomic price data: True or False
INCLUDE_FRED_DATA = True

# ============================================================================

In [ ]:
import os
import re
import sys
import time
import json
import calendar
import random
import logging
from pathlib import Path
from datetime import datetime

import cloudscraper
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from fredapi import Fred
from serpapi import GoogleSearch
from tqdm.auto import tqdm

import torch
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
from sklearn.decomposition import PCA

load_dotenv()
logging.basicConfig(level=logging.WARNING)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

FRED_API_KEY = os.getenv('FRED')
SERP_API_KEYS = []
for key_name in ['SERP_API_1', 'SERP_API_2', 'SERP_API_3']:
    k = os.getenv(key_name)
    if k:
        SERP_API_KEYS.append(k)

START_DATE = '2010-01-01'
END_DATE   = '2026-03-31'
YEARS      = list(range(2010, 2027))

# Commodity configurations
COMMODITY_CONFIG = {
    'wheat': {
        'fred_series': 'PWHEAMTUSDM',
        'fred_frequency': 'monthly',
        'serp_queries': ['wheat'],
        'base_url': 'https://www.investing.com/commodities/us-wheat-news',
        'last_page': 46,
    },
    'corn': {
        'fred_series': 'PMAIZMTUSDM',
        'fred_frequency': 'monthly',
        'serp_queries': ['corn'],
        'base_url': 'https://www.investing.com/commodities/us-corn-news',
        'last_page': 46,
    },
    'oil': {
        'fred_series': 'DCOILWTICO',
        'fred_frequency': 'daily',
        'serp_queries': ['oil'],
        'base_url': 'https://www.investing.com/commodities/crude-oil-news',
        'last_page': 46,
    },
}

# Create commodity data directory
(DATA_DIR / SELECTED_COMMODITY).mkdir(exist_ok=True)

MODEL_NAME = "ProsusAI/finbert"
MAX_LEN    = 512
BATCH_SIZE = 8
PCA_DIM    = 16

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"\nConfiguration:")
print(f"  Commodity: {SELECTED_COMMODITY.upper()}")
print(f"  News Type: {SELECTED_NEWS_TYPE.upper()}")
print(f"  Include FRED: {INCLUDE_FRED_DATA}")

## Part 1: News Scraper Helpers
Functions for fetching data from Investing.com payloads and SERP API.

In [ ]:
def make_scraper():
    import cloudscraper
    return cloudscraper.create_scraper(
        browser={'browser': 'chrome', 'platform': 'darwin', 'mobile': False}
    )

def page_url(base_url: str, page: int) -> str:
    if page == 1:
        return base_url
    return f'{base_url}/{page}'

def fetch_page(scraper, url: str, retries: int = 3):
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.investing.com/',
    }
    for attempt in range(1, retries + 1):
        try:
            r = scraper.get(url, headers=headers, timeout=30)
            if r.status_code == 200:
                return BeautifulSoup(r.text, 'lxml')
            elif r.status_code == 404:
                return None
        except Exception:
            pass
        if attempt < retries:
            time.sleep(2 ** attempt + random.uniform(0, 2))
    return None

def parse_articles_investing(soup, commodity: str, page: int) -> list:
    records = []
    for art in soup.find_all('article', attrs={'data-test': 'article-item'}):
        title_tag = art.find('a', attrs={'data-test': 'article-title-link'})
        title = title_tag.get_text(strip=True) if title_tag else ''
        url = title_tag['href'] if title_tag else ''
        if url and not url.startswith('http'):
            url = 'https://www.investing.com' + url
        desc_tag = art.find('p', attrs={'data-test': 'article-description'})
        description = desc_tag.get_text(strip=True) if desc_tag else ''
        src_tag = art.find('a', attrs={'data-test': 'article-provider-link'})
        source = src_tag.get_text(strip=True) if src_tag else 'investing.com'
        date_tag = art.find('time', attrs={'data-test': 'article-publish-date'})
        date = date_tag.get('datetime', date_tag.get_text(strip=True)) if date_tag else ''
        if title:
            records.append({
                'commodity': commodity, 'title': title, 'date': date,
                'source': source, 'description': description, 'url': url
            })
    return records

def already_scraped_pages(csv_path) -> set:
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path, on_bad_lines='skip', engine='python')
            if 'page_scraped' in df.columns:
                return set(df['page_scraped'].dropna().astype(int).tolist())
        except Exception:
            pass
    return set()

def append_to_csv(records: list, csv_path):
    if not records:
        return
    try:
        df_new = pd.DataFrame(records)
        write_header = not csv_path.exists()
        df_new.to_csv(csv_path, mode='a', header=write_header, index=False)
    except Exception as e:
        print(f"  [append_to_csv] ERROR: {str(e)}")

def scrape_direct_investing(commodity, base_url, last_page):
    news_csv = DATA_DIR / commodity / f'{commodity}_news.csv'
    done_pages = already_scraped_pages(news_csv)
    todo_pages = [p for p in range(1, last_page + 1) if p not in done_pages]

    if not todo_pages:
        print(f'  [{commodity}] All {last_page} pages already scraped. CSV: {news_csv}')
        return []

    print(f'  [{commodity}] Scraping {len(todo_pages)} pages (skipping {len(done_pages)} already done)...')
    scraper = make_scraper()
    buffer = []
    errors = []

    for i, page in enumerate(tqdm(todo_pages, desc=f"Direct {commodity}", unit='page'), start=1):
        url = page_url(base_url, page)
        soup = fetch_page(scraper, url)

        if soup is None:
            errors.append(page)
            print(f'  [{commodity}] FAILED page {page} — skipping')
        else:
            records = parse_articles_investing(soup, commodity, page)
            for r in records:
                r['page_scraped'] = page
            buffer.extend(records)

        if i % 10 == 0 and buffer:
            append_to_csv(buffer, news_csv)
            buffer.clear()

        if i < len(todo_pages):
            time.sleep(random.uniform(2.0, 4.5))

    if buffer:
        append_to_csv(buffer, news_csv)

    return []

In [ ]:
def scrape_payload_news(commodity):
    payload_file = DATA_DIR / commodity / f'{commodity}_payload.json'
    ckpt_file = DATA_DIR / commodity / f'{commodity}_payload_ckpt.json'
    news_csv = DATA_DIR / commodity / f'{commodity}_news.csv'
    
    if not payload_file.exists(): 
        return []
    
    data = json.loads(payload_file.read_text())
    items = data.get('news', []) + data.get('articles', [])
    records = []
    fetched_urls = set()
    
    if ckpt_file.exists():
        records = json.loads(ckpt_file.read_text())
        fetched_urls.update(r['url'] for r in records)
        
    if news_csv.exists():
        try:
            df_existing = pd.read_csv(news_csv, on_bad_lines='skip', engine='python')
            if 'url' in df_existing.columns:
                fetched_urls.update(df_existing['url'].dropna().tolist())
        except Exception:
            pass
    
    scraper = cloudscraper.create_scraper()
    for item in tqdm(items, desc=f"Payload {commodity}"):
        url = item.get('link', '')
        if not url: 
            continue
        if url.startswith('/'):
            url = 'https://www.investing.com' + url
            
        if url in fetched_urls:
            continue
            
        text = ""
        try:
            resp = scraper.get(url, timeout=10)
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, 'html.parser')
                body = soup.find('div', class_='WYSIWYG articlePage') or soup.find('div', class_='article_container')
                if body:
                    text = ' '.join([p.get_text(strip=True) for p in body.find_all('p')])
            time.sleep(2.0)
        except Exception:
            time.sleep(2.0)
            
        ts = item.get('dateTimestamp')
        if ts and isinstance(ts, int):
            dt = datetime.fromtimestamp(ts, datetime.UTC).strftime('%Y-%m-%d')
        else:
            dt = item.get('date', '')
            
        records.append({
            'commodity': commodity,
            'title': item.get('name', item.get('title', '')),
            'date': dt,
            'source': item.get('providerName', 'Investing.com'),
            'description': text if text else item.get('content', ''),
            'url': url
        })
        fetched_urls.add(url)
        
        if len(records) % 50 == 0:
            ckpt_file.write_text(json.dumps(records))
            
    ckpt_file.write_text(json.dumps(records))
    return records

In [ ]:
CHECKPOINT_FILE = DATA_DIR / 'serp_checkpoint_bloomberg_monthly.json'
current_key_idx = 0

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        return json.loads(CHECKPOINT_FILE.read_text())
    return {'fetched': {}}

def save_checkpoint(ckpt):
    CHECKPOINT_FILE.write_text(json.dumps(ckpt, indent=2))

ckpt = load_checkpoint()

def fetch_bloomberg_monthly(commodity, query, year, month, ckpt):
    global current_key_idx
    
    key = f'{commodity}:{query}:{year}:{month:02d}'
    if key in ckpt['fetched']: 
        return []
    
    if current_key_idx >= len(SERP_API_KEYS): 
        return []
    
    records = []
    page = 0
    last_day = calendar.monthrange(year, month)[1]
    
    while True:
        params = {
            'q': f'{query} site:bloomberg.com',
            'api_key': SERP_API_KEYS[current_key_idx],
            'num': 100,
            'start': page * 100,
            'tbs': f'cdr:1,cd_min:{month:02d}/01/{year},cd_max:{month:02d}/{last_day:02d}/{year}',
            'tbm': 'nws'
        }
        
        try:
            result = GoogleSearch(params).get_dict()
            
            if 'error' in result:
                if 'run out' in result['error'].lower() or 'exhausted' in result['error'].lower():
                    current_key_idx += 1
                    if current_key_idx >= len(SERP_API_KEYS):
                        break
                    continue
                else:
                    break
            
            organic = result.get('news_results', []) or result.get('organic_results', [])
            if not organic: 
                break
                
            for r in organic:
                date_str = r.get('date', f'{year}-{month:02d}-01')
                records.append({
                    'commodity': commodity,
                    'title': r.get('title', ''),
                    'date': date_str,
                    'source': 'Bloomberg',
                    'description': r.get('snippet', ''),
                    'url': r.get('link', ''),
                })
                
            if len(organic) < 100: 
                break
            page += 1
            time.sleep(2.0)
            
        except Exception:
            break
            
    ckpt['fetched'][key] = len(records)
    save_checkpoint(ckpt)
    return records

## Part 2: Main Scraping Function
Orchestrates scraping based on selected news type and deduplicates results.

In [ ]:
def parse_serp_date(raw):
    if not raw: 
        return ''
    if 'ago' in raw.lower(): 
        return datetime.utcnow().strftime('%Y-%m-%d')
    _MONTHS = {m: i+1 for i, m in enumerate(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])}
    m = re.search(r'([A-Za-z]+)\s+(\d{1,2}),?\s+(\d{4})', raw)
    if m:
        mon_str, day, year = m.group(1)[:3].capitalize(), int(m.group(2)), int(m.group(3))
        mon = _MONTHS.get(mon_str, 0)
        if mon: 
            return f'{year}-{mon:02d}-{day:02d}'
    m2 = re.search(r'(\d{4})-(\d{2})-(\d{2})', raw)
    if m2: 
        return m2.group(0)
    return raw

def run_scraper_and_clean(commodity, news_type):
    cfg = COMMODITY_CONFIG.get(commodity)
    if not cfg:
        print(f"ERROR: Commodity '{commodity}' not supported. Choose from: {', '.join(COMMODITY_CONFIG.keys())}")
        return
    
    print(f"\n[INFO] Starting news collection for {commodity.upper()} ({news_type} method)...")
    
    all_records = []
    
    if news_type in ['direct', 'all']:
        print(f"  [1/3] Scraping directly from Investing.com for {commodity}...")
        direct_records = scrape_direct_investing(commodity, cfg['base_url'], cfg['last_page'])
        all_records.extend(direct_records)
    
    if news_type in ['payload', 'all']:
        print(f"  [2/3] Scraping Investing.com payloads for {commodity}...")
        payload_records = scrape_payload_news(commodity)
        all_records.extend(payload_records)
    
    if news_type in ['bloomberg', 'all']:
        print(f"  [3/3] Fetching Bloomberg news via SERP API for {commodity}...")
        bloomberg_records = []
        for query in cfg['serp_queries']:
            for year in YEARS:
                for month in range(1, 13):
                    if current_key_idx < len(SERP_API_KEYS):
                        res = fetch_bloomberg_monthly(commodity, query, year, month, ckpt)
                        bloomberg_records.extend(res)
        all_records.extend(bloomberg_records)
    
    news_csv = DATA_DIR / commodity / f'{commodity}_news.csv'
    df_existing = pd.DataFrame()
    if news_csv.exists():
        df_existing = pd.read_csv(news_csv, on_bad_lines='skip', engine='python')
    
    df_new = pd.DataFrame(all_records)
    if not df_new.empty:
        df_new['date'] = df_new['date'].apply(parse_serp_date)
    
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    if not df_combined.empty:
        df_combined['date'] = pd.to_datetime(df_combined['date'], errors='coerce')
        df_combined = df_combined.dropna(subset=['date'])
        df_combined = df_combined.sort_values('date', ascending=False)
        
        initial_len = len(df_combined)
        df_combined = df_combined.drop_duplicates(subset=['source', 'title'], keep='first')
        df_combined = df_combined.sort_values("date")
        df_combined.to_csv(news_csv, index=False)
        print(f"✓ FINAL: {len(df_combined)} unique {commodity} articles saved (removed {initial_len - len(df_combined)} duplicates)")
    else:
        print(f"✓ No new articles to save for {commodity}")

In [ ]:
run_scraper_and_clean(SELECTED_COMMODITY, SELECTED_NEWS_TYPE)

## Part 3: FRED Pricing (Conditional)
Collects historical commodity prices using the FRED API if enabled.

In [ ]:
def fetch_fred_prices(commodity, start, end):
    cfg = COMMODITY_CONFIG.get(commodity)
    if not cfg:
        print(f"ERROR: Commodity '{commodity}' not supported.")
        return pd.DataFrame()
    
    if not FRED_API_KEY:
        print(f"ERROR: FRED_API_KEY not found in environment variables.")
        return pd.DataFrame()
    
    fred = Fred(api_key=FRED_API_KEY)
    series_id = cfg['fred_series']
    frequency = cfg['fred_frequency']
    
    try:
        raw = fred.get_series(series_id, observation_start=start, observation_end=end)
    except Exception as e:
        print(f"ERROR fetching FRED series {series_id}: {e}")
        return pd.DataFrame()
    
    df = pd.DataFrame({'Price': raw}).reset_index()
    df.columns = ['Date', 'Price']
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.dropna(subset=['Price']).sort_values('Date')

    if frequency == 'monthly':
        df = df.set_index('Date').resample('B').ffill().reset_index()

    df = df[(df['Date'] >= start) & (df['Date'] <= end)]
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    return df.reset_index(drop=True)

if INCLUDE_FRED_DATA:
    print(f"\n[INFO] Fetching FRED prices for {SELECTED_COMMODITY.upper()}...")
    df_prices = fetch_fred_prices(SELECTED_COMMODITY, START_DATE, END_DATE)
    if not df_prices.empty:
        out_csv = DATA_DIR / SELECTED_COMMODITY / f'{SELECTED_COMMODITY}_prices.csv'
        df_prices.to_csv(out_csv, index=False)
        print(f"✓ FRED Prices for {SELECTED_COMMODITY.upper()}: {len(df_prices)} records saved")
    else:
        print(f"✗ Failed to fetch FRED prices for {SELECTED_COMMODITY.upper()}")
else:
    print(f"\n[INFO] Skipping FRED data (INCLUDE_FRED_DATA=False)")

## Part 4: FinBERT Embeddings & Sentiment Pipeline
Extracts embeddings and sentiment from the collected news articles.

In [ ]:
_BOILERPLATE_RE = re.compile(
    r"^(?:\*\s*)?(?:By\s+[A-Z][a-zA-Z\s\-']+"
    r"[A-Z]{2,}[\w\s,]*\([^)]+\)\s*[-–—]\s*)?(?:\*\s*)?",
    re.MULTILINE,
)

def clean_text(row):
    title = str(row.get("title", "")).strip()
    description = str(row.get("description", "")).strip()
    
    description = _BOILERPLATE_RE.sub("", description).strip()
    if description.endswith("..."):
        description = description[:-3].strip()
        
    if description:
        return f"{title}. {description}"
    return title

def extract_cls_embeddings(texts, tokenizer, model, device, batch_size=BATCH_SIZE):
    all_embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Extracting embeddings", leave=False):
        batch_texts = texts[start : start + batch_size]
        encoded = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

def apply_pca(embeddings, n_components=PCA_DIM):
    embeddings_np = embeddings.numpy()
    pca = PCA(n_components=n_components)
    reduced_embeddings_np = pca.fit_transform(embeddings_np)
    return torch.from_numpy(reduced_embeddings_np).float()

def extract_sentiment_scores(texts, tokenizer, sentiment_model, device, batch_size=BATCH_SIZE):
    all_scores = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Scoring sentiment", leave=False):
        batch_texts = texts[start : start + batch_size]
        encoded = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            logits = sentiment_model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).cpu()
            
        for row in probs:
            pos, neg, neu = row[0].item(), row[1].item(), row[2].item()
            all_scores.append({
                "sentiment_pos": pos,
                "sentiment_neg": neg,
                "sentiment_neu": neu,
                "sentiment_score": pos - neg,
            })
    return pd.DataFrame(all_scores)

def aggregate_daily(df, embeddings):
    dates = df["date_day"].tolist()
    daily_map = {}
    for idx, d in enumerate(dates):
        daily_map.setdefault(d, []).append(idx)
        
    result = {}
    for day, indices in sorted(daily_map.items()):
        stacked = embeddings[indices]
        result[day] = torch.mean(stacked, dim=0)
    return result

## Part 5: Execute Embeddings & Sentiment Pipeline
Runs the embedding pipeline for the selected commodity.

In [ ]:
def run_embedding_pipeline(commodity):
    print(f"\n[INFO] Loading tokenizer & models...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
    sent_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()

    news_csv = DATA_DIR / commodity / f'{commodity}_news.csv'
    out_pt = DATA_DIR / commodity / f'{commodity}_news_embeddings.pt'
    out_csv = DATA_DIR / commodity / f'{commodity}_news_sentiment.csv'
    
    if not news_csv.exists():
        print(f"✗ News file not found: {news_csv}")
        return
        
    print(f"\n[INFO] Processing {commodity.upper()}...")
    df = pd.read_csv(news_csv, on_bad_lines='skip', engine='python')
    
    n_before = len(df)
    df = df.drop_duplicates(subset=["title", "source"], keep="first")
    n_after = len(df)
    if (n_before - n_after) > 0:
        print(f"  Removed {n_before - n_after} duplicate(s)")
        
    df = df.reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])
    df["date_day"] = df["date"].dt.strftime("%Y-%m-%d")
    df["clean_text"] = df.apply(clean_text, axis=1)
    
    texts = df["clean_text"].tolist()
    
    embeddings = extract_cls_embeddings(texts, tokenizer, base_model, device, BATCH_SIZE)
    if embeddings.shape[1] > PCA_DIM:
        embeddings = apply_pca(embeddings, PCA_DIM)
        
    sent_df = extract_sentiment_scores(texts, tokenizer, sent_model, device, BATCH_SIZE)
    df = pd.concat([df.reset_index(drop=True), sent_df], axis=1)
    
    daily_embeddings = aggregate_daily(df, embeddings)
    
    daily_sent = (df.groupby("date_day")[["sentiment_pos", "sentiment_neg", "sentiment_neu", "sentiment_score"]]
                    .mean()
                    .reset_index()
                    .rename(columns={"date_day": "date"}))
    daily_sent["article_count"] = df.groupby("date_day").size().values
    daily_sent = daily_sent.sort_values("date").reset_index(drop=True)
    
    torch.save(daily_embeddings, out_pt)
    daily_sent.to_csv(out_csv, index=False)
    
    print(f"✓ Embeddings saved to: {out_pt}")
    print(f"✓ Sentiment data saved to: {out_csv}")
    print(f"  Total sentiment records: {len(daily_sent)}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

run_embedding_pipeline(SELECTED_COMMODITY)

## Part 6: Summary
Displays final dataset statistics.

In [ ]:
print("\n" + "═" * 60)
print(f"FINAL SUMMARY FOR {SELECTED_COMMODITY.upper()}")
print("═" * 60)

try:
    news_len = len(pd.read_csv(DATA_DIR / SELECTED_COMMODITY / f'{SELECTED_COMMODITY}_news.csv', on_bad_lines='skip', engine='python'))
except:
    news_len = 0

try:
    sent_len = len(pd.read_csv(DATA_DIR / SELECTED_COMMODITY / f'{SELECTED_COMMODITY}_news_sentiment.csv'))
except:
    sent_len = 0

try:
    if INCLUDE_FRED_DATA:
        fred_len = len(pd.read_csv(DATA_DIR / SELECTED_COMMODITY / f'{SELECTED_COMMODITY}_prices.csv'))
    else:
        fred_len = 0
except:
    fred_len = 0

print(f"{SELECTED_COMMODITY.upper()}:")
print(f"  News Articles:      {news_len} rows")
print(f"  Sentiment Records:  {sent_len} rows")
if INCLUDE_FRED_DATA:
    print(f"  FRED Prices:        {fred_len} rows")
print("═" * 60)